In [1]:
import os
import sys
import pickle
import numpy as np
from numba import njit
import itertools as itt
import aerosandbox as asb

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from Aircraft.Planform import Planform
from Aircraft.Fixed import Fixed
from Drag.Fuselage import Fuselage
from Drag.Bay import Bay
from Drag.LandingGear import LandingGear
from Aircraft.Aircraft import Aircraft
from global_parameters import Assumptions
from Requirements.FuelReq import FuelReq
from Requirements.LGReq import LGReq
from Requirements.MassReq import MassReq
from Requirements.MDReq import MDReq
from Requirements.EmpennageReq import EmpennageReq
from Requirements.Requirement import Requirement
from EmpennageSizing.TailFinder import TailFinder
from EmpennageSizing.CanardFinder import CanardFinder
from structural_analysis.Material import Material

# Loading the pre-computed planforms and the fuselage

In [2]:
with open("pickles/planform_pickle_official.pcl", "r+b") as f:
    plaforms_recovered:list[tuple[Planform, str, bool]] = pickle.load(f)

assumptions = Assumptions()

In [3]:
# # --- Import Onshape pull utilities ---
# sys.path.append(os.path.abspath(os.getcwd()))
# from onshape_pull import (
#     fetch_variable_studio, fetch_measurement_features,
#     evaluate_measurements, load_cached_masses, fetch_mass_properties,
#     compute_cg_scenarios, lookup_var, lookup_meas,
#     UPDATE_MASSES,
# )

# # --- Pull data from Onshape ---
# variables = fetch_variable_studio()
# meas_names = fetch_measurement_features()
# measurements = evaluate_measurements(meas_names)
# components = load_cached_masses() if not UPDATE_MASSES else fetch_mass_properties()
# cg_data = compute_cg_scenarios(components)

# # --- Z offset (axle datum) ---
# Front_Landing_Gear_Hinge_Z = lookup_meas(measurements, "Front_Landing_Gear_Hinge_Z")
# Front_Strut_Height, _, _ = lookup_var(variables, "Front_Strut_Height")
# Front_Gear_Extension_Max = lookup_meas(measurements, "Front_Gear_Extension_Max")
# Front_Gear_Extension_Min = lookup_meas(measurements, "Front_Gear_Extension_Min")
# z_offset = (Front_Landing_Gear_Hinge_Z + Front_Strut_Height
#             + (Front_Gear_Extension_Max - Front_Gear_Extension_Min))

# # --- Build drag components ---
# engine_bay = Bay(
#     surface_wetted=83744.32631 / 1e6,  # mm² → m² (hardcoded, not in Onshape)
#     length=0.172,                       # 172 mm (hardcoded, not in Onshape)
#     diameter=lookup_var(variables, "engine_diameter")[0],
# )

# Front_Gear_Unexposed = lookup_meas(measurements, "Front_Gear_Unexposed")
# nose_gear = LandingGear(
#     wheel_width=0.025,
#     exposed_height=Front_Strut_Height - Front_Gear_Unexposed,
#     wheel_diameter=lookup_var(variables, "Wheel_Diameter")[0],
#     strut_width=lookup_var(variables, "Front_Strut_Diameter")[0],
# )

# Rear_Strut_Height, _, _ = lookup_var(variables, "Rear_Strut_Height")
# Rear_Strut_height_2, _, _ = lookup_var(variables, "Rear_Strut_height_2")
# main_gear = LandingGear(
#     wheel_width=0.025,
#     exposed_height=Rear_Strut_Height + Rear_Strut_height_2,
#     wheel_diameter=lookup_var(variables, "Wheel_Diameter")[0],
#     strut_width=lookup_var(variables, "Rear_Strut_Diameter")[0],
# )

# fuselage = Fuselage(
#     surface_wetted=lookup_meas(measurements, "Wetted_Area"),
#     length_total=lookup_var(variables, "FuselageLength")[0],
#     diameter_max=lookup_var(variables, "FuselageHeight")[0],
#     upsweep=0.0,
#     base_area=lookup_meas(measurements, "Base_Area"),
# )

# # --- X-position helpers ---
# WingPortDistance, _, _ = lookup_var(variables, "FuselageLength")
# WingPortDistance = (WingPortDistance / 2) - 0.025
# WingPortWidth, _, _ = lookup_var(variables, "WingPortWidth")
# CanardPortXLoc, _, _ = lookup_var(variables, "CanardPortXLoc")
# CanardPortWidth, _, _ = lookup_var(variables, "CanardPortWidth")

# print(cg_data["x_cg_min"])
# print(cg_data["x_cg_max"])


# # --- Fixed parameters ---
# fixed = Fixed(
#     mass=cg_data["mass"],
#     fuel_mass=cg_data["fuel_mass"],
#     x_cg_min=cg_data["x_cg_min"],
#     x_cg_max=cg_data["x_cg_max"],
#     x_tail_cone=lookup_meas(measurements, "Tailcone_X"),
#     z_cg=cg_data["z_cg_full"] + z_offset,
#     z_tail_cone=-lookup_meas(measurements, "Z_TailCone") + z_offset,
#     z_wing=lookup_meas(measurements, "Z_wing_LE_Abs") + z_offset,
#     x_LE_canard=CanardPortXLoc + CanardPortWidth / 2,
#     x_LE_wing=WingPortDistance + 0.115 - int(WingPortWidth) / 2,
#     x_LE_tail=lookup_meas(measurements, "X_LE_Tail"),
#     x_nose_gear=lookup_meas(measurements, "x_nose_gear"),
#     x_main_gear=lookup_meas(measurements, "x_main_gear"),
#     y_main_gear=0.419,
#     fuselage=fuselage,
#     nose_gear=nose_gear,
#     main_gear=main_gear,
#     engine_bay=engine_bay,
# )

In [4]:
# with open("pickles/fixed_pickle.pcl", "wb") as f:
#     pickle.dump(fixed, f)

In [5]:
with open("pickles/fixed_pickle.pcl", "rb") as f:
    fixed:Fixed = pickle.load(f)

In [6]:
print(fixed.x_cg_max, fixed.x_cg_min, fixed.x_LE_wing, fixed.z_cg, fixed.z_tail_cone)
fixed.x_LE_wing = 1.255
delta_z = 0.01
fixed.z_tail_cone += delta_z
fixed.z_cg += delta_z
print(fixed.z_cg, fixed.z_tail_cone)

1.3589152301106575 1.271545179962353 1.4400000000000002 0.17957635338517247 0.11000000000000004
0.18957635338517248 0.12000000000000004


In [7]:
for component in fixed.drag_components(False):
    component.add_cache_entry("go_around", assumptions.airspeed_approach/asb.Atmosphere(assumptions.altitude_go_round).speed_of_sound(), assumptions.altitude_go_round)
    component.add_cache_entry("mach_max", assumptions.mach_max, assumptions.altitude_mach_max)
    component.add_cache_entry("cruise", assumptions.mach_cruise, assumptions.altitude_cruise)
for component in fixed.drag_components(True):
    component.add_cache_entry("takeoff", assumptions.airspeed_approach/asb.Atmosphere().speed_of_sound(), 0.)

# Creating full Aircraft objects

In [8]:
material_skin = Material(assumptions.cfrp_density, elastic_modulus=assumptions.cfrp_Young_modulus, 
                         poisson_ratio=assumptions.cfrp_poisson, shear_modulus=assumptions.cfrp_Young_modulus / 2 / (1 + assumptions.cfrp_poisson),
                         yield_strength=assumptions.cfrp_yield_strength, fracture_strength=assumptions.cfrp_yield_strength)

In [9]:
aircraft:list[Aircraft] = list()

for i, planform_recovered in enumerate(plaforms_recovered):
    main_wing = planform_recovered[0]
    planform_type = planform_recovered[1]

    ef = TailFinder(fixed, material=material_skin, core_density=assumptions.foam_denisty, thicknesses=assumptions.allowable_thicknesses, safety_factor=assumptions.structural_safety_factor, AR_h=max(4., main_wing.aspect_ratio/2)) if (planform_type == "tail") else CanardFinder(fixed, material=material_skin, core_density=assumptions.foam_denisty, thicknesses=assumptions.allowable_thicknesses, safety_factor=assumptions.structural_safety_factor, AR_c=max(5., main_wing.aspect_ratio/2))
    
    emp = ef.find_planforms(main_wing, print_=i==28)

    for e in emp:
        e.add_cache_entry("go_around", assumptions.airspeed_approach/asb.Atmosphere(assumptions.altitude_go_round).speed_of_sound(), assumptions.altitude_go_round)
        e.add_cache_entry("mach_max", assumptions.mach_max, assumptions.altitude_mach_max)
        e.add_cache_entry("cruise", assumptions.mach_cruise, assumptions.altitude_cruise)
        e.add_cache_entry("takeoff", assumptions.airspeed_approach/asb.Atmosphere().speed_of_sound(), 0.)
        e.mass_cache = 1
        e.x_cg_cache = .1

    aircraft_planforms = [main_wing] + emp #TODO add the empenage
    aircraft.append(Aircraft(
        fixed=fixed, #TODO: add the fuselage from CAD
        planforms=aircraft_planforms 
    ))

Stresses 24013280.215892255, 73961606.64878033, 0.0004
Stresses 11968503.891237225, 40305038.507069916, 0.0007999999999999999
Stresses 72192385.51451235, 800706851.155794, 0.0004
Stresses 36058056.54054728, 531013372.57275385, 0.0007999999999999999
Stresses 24013280.2158923, 471266374.5731791, 0.0012
Stresses 17990892.05356475, 462211403.1722321, 0.0015999999999999999
Stresses 14377459.15616823, 457575158.55498266, 0.002
Stresses 11968503.891237248, 426573347.70427364, 0.0024000000000000002
Stresses 10247821.559143653, 360303215.76490134, 0.0028
Stresses 24681691.009495843, 76019323.91275515, 0.0004
Stresses 12301643.332592154, 41322003.72360744, 0.0007999999999999999
Stresses 70217256.50920288, 722698502.9456797, 0.0004
Stresses 35069426.08244567, 468049174.8049482, 0.0007999999999999999
Stresses 23353482.60685999, 406223267.40172315, 0.0012
Stresses 17495510.869067065, 392911057.56442684, 0.0015999999999999999
Stresses 24677645.61121085, 75823031.64053106, 0.0004
Stresses 12299627.08

In [10]:
s_ratios = [ac.planforms[1].wing_area / ac.planforms[0].wing_area for ac in aircraft]
print(s_ratios)
print(min(s_ratios), np.average(s_ratios), max(s_ratios))
ac_bad:Aircraft = aircraft[np.argmax(s_ratios)],
print(np.argmax(s_ratios))
ac_bad= ac_bad[0]
print(ac_bad.planforms[0].sweep_quarter_rad, ac_bad.planforms[0].aspect_ratio, ac_bad.planforms[0].cm_quarter_chord, ac_bad.planforms[0].thickness_to_chord)

[0.10563379177039287, 0.11884674026919968, 0.04955529343646527, 0.018598035911128404, 0.24863986729264517, 0.25768833961577114, 0.15850375306165254, 0.1589876211699311, 0.1055307190250254, 0.11872816010820748, 0.049644096153517926, 0.018481514893502797, 0.2485337438372989, 0.2575694009711863, 0.15839525428918136, 0.158871803351454, 0.10535531402160579, 0.11852611591010388, 0.04979538002144742, 0.018282985341441182, 0.24835291472306065, 0.2573667498237613, 0.15821037618514455, 0.15867447121835976, 0.0743312813579385, 0.0737732254142741, 0.07610418483409993, 0.00031523344479899125, 0.2525383231539187, 0.269358309006963, 0.19024837265424044, 0.19850819578554557, 0.07444954654310289, 0.07360835305412608, 0.07444954654310289, 0.001944417293204241, 0.25239562959454565, 0.26919340869661396, 0.1922761296964297, 0.20077706004126755, 0.07465102609650097, 0.07332745593463404, 0.07465102609650097, 0.0016690077488530257, 0.25215249935520073, 0.26891246833694593, 0.19202938043427598, 0.2005008333131

# Checking if reuirements are met

In [11]:
for ac in aircraft:
    ac.fixed = fixed

In [12]:
requirements:list[Requirement] = [
    MassReq(50.),
    MDReq(),
    FuelReq(),
    LGReq(),
    EmpennageReq(),
]

requirement_labels = [
    "MTOM",
    "Matching Diagram",
    "Fuel",
    "Landing Gear",
    "Empennage Requirement"
]

In [13]:
for ac in aircraft:
    failed_reqs = list()
    for requirement, label in zip(requirements, requirement_labels):
        if type(requirement) == LGReq:
                print("f")
        if not requirement.assess(ac):
            failed_reqs.append(label)

    if len(failed_reqs):
        print(f"ac mass: {ac.total_mass()}, {ac.planforms[0].oswald}")
        print(f"MainWing: AR={ac.planforms[0].aspect_ratio}, tc={ac.planforms[0].thickness_to_chord}, sweep={np.rad2deg(ac.planforms[0].sweep_quarter_rad)} deg, cmac={ac.planforms[0].cm_quarter_chord}")
        print(f"Failed: {failed_reqs}")
        print()

Fuel available: 15.0457919996796 kg
Fuel required: 6.484545863143035 kg
Difference: 8.561246136536566 kg
f
all constraints satisfied
Fuel available: 15.0457919996796 kg
Fuel required: 6.465959013508825 kg
Difference: 8.579832986170775 kg
f
all constraints satisfied
Fuel available: 15.0457919996796 kg
Fuel required: 6.492836776189709 kg
Difference: 8.552955223489892 kg
f
all constraints satisfied
Fuel available: 15.0457919996796 kg
Fuel required: 6.477142703637059 kg
Difference: 8.568649296042542 kg
f
all constraints satisfied
Fuel available: 15.0457919996796 kg
Fuel required: 6.58453336488153 kg
Difference: 8.461258634798071 kg
f
all constraints satisfied
Fuel available: 15.0457919996796 kg
Fuel required: 6.57783627158082 kg
Difference: 8.467955728098781 kg
f
all constraints satisfied
Fuel available: 15.0457919996796 kg
Fuel required: 6.604306590698863 kg
Difference: 8.441485408980737 kg
f
all constraints satisfied
Fuel available: 15.0457919996796 kg
Fuel required: 6.590162916841198 kg